In [ ]:
#
# This script extracts annotated information from UIMA XMI files stored in ZIP archives 
# and converts them into structured JSON documents for downstream processing or analysis.
#

from zipfile import ZipFile
from cassis import load_typesystem, load_cas_from_xmi
from glob import glob
import json
from pathlib import Path
from pydantic import BaseModel
from typing import Optional, List
import random
import re



# Set a fixed seed for reproducibility
random.seed(42)

# Pydantic models for annotations and documents
class CardioScoreAnnotation(BaseModel):
    reasoning: Optional[str]
    entity_type: Optional[str]
    typ: Optional[str]
    aktualitaet: Optional[str]
    numeral: Optional[float]
    unit: Optional[str]
    scores: Optional[str]
    annotated_span: str
    begin: int
    end: int

class CardioScoreDocument(BaseModel):
    text: str
    annotations: List[CardioScoreAnnotation]

# Input and output paths
data_path = "data/raw/extracted_corpus"
output_dir = Path("data/json/annotations")
output_dir.mkdir(parents=True, exist_ok=True)

# Find all .zip files recursively
dataset = glob(f"{data_path}/**/*.zip", recursive=True)


def open_xmi(file_path):
    """
    Opens a ZIP archive, loads the UIMA TypeSystem.xml and the first .xmi CAS file.
    Returns (typesystem, cas) or (None, None) on failure.
    """
    try:
        with ZipFile(file_path, 'r') as zf:
            names = zf.namelist()
            # Locate TypeSystem.xml
            ts_name = next((n for n in names if n.endswith('TypeSystem.xml')), None)
            if ts_name is None:
                print(f"⚠️  Missing TypeSystem.xml in {file_path}")
                return None, None
            with zf.open(ts_name) as f:
                typesystem = load_typesystem(f)
            # Locate the CAS XMI file
            xmi_name = next((n for n in names if n.endswith('.xmi') and 'TypeSystem' not in n), None)
            if xmi_name is None:
                print(f"⚠️  No XMI file found in: {file_path}")
                return None, None
            with zf.open(xmi_name) as f:
                cas = load_cas_from_xmi(f, typesystem=typesystem)
            return typesystem, cas
    except Exception as e:
        print(f"Error opening {file_path}: {e}")
        return None, None

# Process each ZIP archive to JSON
for zip_file in dataset:
    typesystem, cas = open_xmi(zip_file)
    if typesystem is None or cas is None:
        continue

    # Extract the document text
    text = ""
    for view in cas.views:
        if view.sofa and view.sofa.sofaString:
            text = view.sofa.sofaString
            break

    # Determine output filename based on ZIP path
    fp = Path(zip_file).resolve()
    # Extract leading numeric ID for the output filename
    base = fp.stem
    match = re.match(r"(\d+)", base)
    name = match.group(1) if match else base

    # Extract annotations
    tag_type = typesystem.get_type('webanno.custom.CardioScore')
    annotations = []
    for tag in cas.select(tag_type.name):
        # Handle comma as decimal point
        num_str = (tag.Numeral or '').replace(',', '.')
        try:
            numeral = float(num_str) if num_str else None
        except ValueError:
            numeral = None

        annotation = CardioScoreAnnotation(
            reasoning="",
            entity_type=tag.Entity,
            typ=tag.Typ,
            aktualitaet=tag.Aktualitt,
            numeral=numeral,
            unit=tag.Unit,
            scores=tag.Scores,
            annotated_span=text[tag.begin:tag.end],
            begin=tag.begin,
            end=tag.end
        )
        annotations.append(annotation)

    # Build and save JSON
    document = CardioScoreDocument(text=text, annotations=annotations)
    output_path = output_dir / f"{name}.json"
    with open(output_path, 'w', encoding='utf-8') as f:
        doc_dict = document.model_dump()
        json_str = json.dumps(doc_dict, indent=2, ensure_ascii=False)
        f.write(json_str)
    print(f"Saved: {name}.json")
